## Analyzing the Tennessee Education Achievement Data

This notebook is designed to accompany the DS 3100 lectures on data wrangling, exploratory data analysis, and visualization.

### Case-study question

> **What can the 2018 Tennessee education data tell us about differences in achievement across subjects and districts?**

You will progressively build an analysis rather than answer all questions at once.

## 0. The datasets

### Primary dataset: `tenn2018.csv`

The primary file contains 2018 Tennessee achievement results. Each row combines information about a district/school, student subgroup, subject, and achievement percentages, with 2017 baseline measures where available.

Important variables include:

- `district_number`, `district_name`
- `school_number`, `school_name`
- `subgroup`
- `overall_subject`
- `percent_below`, `percent_approaching`, `percent_on_track`, `percent_mastered`
- `percent_on_mastered`
- corresponding `_previous` measures

### Companion dataset: `tn_district_info.csv`

For our introductory join exercise, we will also use a **small lookup table derived from the primary 2018 district file**. It contains one row per district with the number of distinct schools and a simple district-size category. 
In a real analysis, this kind of information would typically come from a separate administrative source and it would often be necessary to strategically merge its information with the primary data for a more robust analysis.

Columns:

- `district_number`
- `district_name`
- `num_schools`
- `district_size`

Official Tennessee data-download information: https://www.tn.gov/education/districts/federal-programs-and-oversight/data/data-downloads.html

### The question guiding our analysis

> **How does student achievement vary across subjects, places, and student populations in Tennessee?**


### 1. Getting to know the data

Before changing anything, inspect the structure of the primary dataset to understand the unit of observation.

::: {.callout-tip}

### Initial Thoughts

1. How many observations and variables are there?
2. What does one row represent?
3. Which variables identify the geographic unit, subgroup, and subject?
4. Which variables are numeric?
5. Which columns contain missing values?

:::

### Some additional context

::: {.callout-note style="font-size:0.75em;"}
## Understanding the level of observation

The Tennessee dataset contains information reported at **multiple geographic levels**. The variables `district_number` and `school_number` help us distinguish between them.

| `district_number` | `school_number` | Represents |
|---|---|---|
| `0` | `0` | Statewide aggregate |
| `> 0` | `0` | District-level aggregate |
| `> 0` | `> 0` | Individual school |

For example, the dataset contains statewide observations where `district_number = 0` and `school_number = 0`, as well as district-level observations such as Anderson County Schools where `district_number = 10` and `school_number = 0`. 
:::

### Explore missingness

`ten.isna()` creates a Boolean DataFrame showing where values are missing. `.sum()` returns the sum of `TRUE` values over the requested axis which is set to 0 by default.

`.any()` checks whether each column contains at least one missing value, and `.sum()` counts how many columns satisfy that condition.

## We need to define the population we want to analyze

The raw data combine multiple **subjects, student subgroups, and geographic levels**. A meaningful comparison requires us to narrow down the population we want to analyze.

::: {.callout-important style="font-size:0.75em;"}

### Your turn

Let's focus our analysis on the following variables:

- school-level observations only (`school_number > 0`);
- `subgroup == "All Students"` rather than specific demography subgroups;
- subjects `ELA` and `Math`;
- the variables needed to compare current and previous performance.

Keep the variables we need:

`district_number`, `district_name`, `school_number`, `school_name`, `overall_subject`, `percent_below`, `percent_on_mastered`, `percent_below_previous`, `percent_on_mastered_previous`.

:::

### Think before coding

Why would calculating a mean on the **entire raw dataset** be misleading?

> We would be mixing subjects, subgroups, and geographic levels. With the context that we have about how data is encoded here, the resulting estimates would be very misleading.


## Understanding the achievement measures

When tracking academic achievement, the state of Tennessee put student scores in one of four performance categories: below grade-level, approaching grade-level, on track, and mastered.

::: {.callout-note style="font-size:0.75em;"}

## Variables tracking student achievement

For our analysis, we will focus primarily on the following variables.

| Variable | Definition |
|---|---|
| `percent_below` | Percentage of students whose 2018 performance falls in the **Below** achievement category |
| `percent_on_mastered` | Percentage of students whose 2018 performance falls in the combined **On Track or Mastered** categories |
| `percent_below_previous` | Percentage of students in the **Below** category in the previous/baseline year |
| `percent_on_mastered_previous` | Percentage of students in the combined **On Track or Mastered** categories in the previous/baseline year |

The dataset identifies **2018** as the current reporting year and generally uses **2017** as the baseline year for these previous-year measures.
:::

### Possible interpretations

- **lower `percent_below`** generally represents a more favorable achievement outcome
- **higher `percent_on_mastered`** represents a higher proportion of students meeting the state's proficiency requirements


## 3. Create variables that help us compare achievement across years

The data include current and previous-year's achievement percentages.  Tracking the changes in these metrics across successive calendar years help us understand potential improvements or regression among different student groups.

::: {.callout-important style="font-size:0.75em;"}

### Your turn

Create two change measures:

- `change_below = percent_below - percent_below_previous`
- `change_on_mastered = percent_on_mastered - percent_on_mastered_previous`

Then inspect their distributions and answer:

- What does a **negative** `change_below` mean?
- What does a **positive** `change_on_mastered` mean?

:::

### Lets recheck the status of missing values

::: {.callout-important style="font-size:0.75em;"}

### Your turn

Why does `change_below` have more missing values than `percent_below`?

:::

Nonetheless it is a good idea to remove these missing values before generating grouped summaries

## 4. Group and summarize

Now lets answer our first analytical question:

> **How do achievement outcomes differ between ELA and Math?**

### Lets recheck the status of missing values

::: {.callout-important style="font-size:0.75em;"}

### Your turn

Calculate, by subject:

- number of available observations;
- mean `percent_below`;
- median `percent_below`;
- mean `percent_on_mastered`;
- mean `change_below`.

Then inspect whether the means and medians tell a similar story.

:::

## 5. Why might we need another dataset?

Suppose we now ask:

> **Do achievement outcomes differ across small, medium, and large districts?**

Look back at the columns in `analysis`.

**Can the primary dataset answer this question directly?**

No: it identifies the district, but it does not contain information regarding district-size.

This is a common reason for a join:

> **The question requires information that lives in another dataset.**

Our companion `tn_district_info.csv` provides that additional information.

## 6. Join the district information

Usually joins happen on a common key shared across datasets. The two tables here share a district identifier:

- `analysis_df_wo_missing`: `district_number`, `district_name`
- `district_info`: `district_number`, `district_name`

::: {.callout-important style="font-size:0.75em;"}

### Your turn

1. Inspect `district_info`.
2. Check whether `district_number` is unique there.
3. Decide which table should be the **left** table.
4. Perform a **left join/merge** so that every achievement observation is retained.
5. Check the number of rows before and after the join.
6. Check whether any observations failed to find district information.

:::

## 7. Post-join wrangling

Now that the district-size information is available, use it to investigate the new question.

::: {.callout-important style="font-size:0.75em;"}

### Your turn

1. Check the number of observations in each `district_size` category.
2. Compare mean `percent_below` across district-size categories.
3. Compare mean `change_below` across district-size categories.
4. Identify any categories with noticeably different results.

:::

Do not interpret differences as **causal**. At this stage we are describing patterns in the data.

The `value_counts()` method in the `pandas` library counts how many times each unique value appears in a column of a dataframe.

In [ ]:
# summary statistics


::: {.callout-tip style="font-size:0.75em;"}

## Food for thought

Why might the **median** tell us something that the mean does not?

:::

## 8. Final wrangling challenge

::: {.callout-important style="font-size:0.75em;"}

## Your turn

Use the joined dataset to answer:

> **For ELA only, how does the average `percent_on_mastered` differ across district-size categories?**

:::



### Closing thoughts

In separate markdown cell(s) explain the following:

- what motivations could we possibly have for filtering to district-level observations;
- why `district_number` and `district_name` form a useful composite key;
- why we chose a left join; 
- what `na.rm = TRUE` does when calculating a summary

::: {.callout-note style="font-size:0.75em;"}

## What's coming next

A summary tells us where the **center** is, but not how the observations are distributed.

Suppose we want to answer questions like:

- How spread out is `percent_below` within each district-size category?
- Are there unusual observations?
- Does the relationship between district size and achievement look consistent across ELA and Math?
- Suppose Math has a certain mean `percent_below`. Do most districts cluster near that mean, or is there substantial variation?

This is where exploring the spread of the data becomes necessary to understand the patterns hiding behind these numbers.<br>

:::

*More to follow!*